# Section 5 - LLM-Based Approach (Claude)

Sentiment classification of the same 1,000 StockEmotions test posts used in Sections 2 and 3, this time with a large language model and no training at all.
The three experiments the brief asks for: zero-shot prompting, few-shot prompting and structured prompting. Same model and settings for all three, only the prompt changes.

Model: Claude Opus 5.5 through the Anthropic API. The model always does a short internal reasoning step before it answers (effort is set to low); the prompts control what it is asked, not whether it reasons.

Needs: a Colab secret called ANTHROPIC_API_KEY (key icon in the left sidebar, notebook access on), the Drive folder COS60011_BERT/data from Section 3, and COS60011_BERT/results/bert_results.csv for the comparison cell.
Outputs go to COS60011_BERT/results_llm: one predictions file per prompt, llm_results.csv, confusion matrices and the comparison chart.

Run order: Sections 1 to 8 once, then Section 9 with QUICK_TEST = True (20 posts, a few cents) to check the key, the prompts and the answer parsing. Then set QUICK_TEST = False in Section 4, re-run Sections 4, 5 and 9 to 12. The full run is 3,000 requests, about 15 minutes and roughly 10 to 30 US dollars of API credit.

## 1. Install the Anthropic SDK

In [ ]:
!pip -q install anthropic

## 2. Mount Google Drive

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
import os

BASE_DIR = "/content/drive/MyDrive/COS60011_BERT"
DATA_DIR = BASE_DIR + "/data"
assert os.path.exists(DATA_DIR + "/stock_emotions_test.csv"), "stock_emotions_test.csv not found in Drive data folder"
print("Drive ready")

## 3. Imports and API client

In [ ]:
import json
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
import anthropic

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=8)
print("anthropic", anthropic.__version__)

## 4. Settings

QUICK_TEST = True scores 20 posts only and writes to results_llm_quick. Set it to False for the real run.

In [ ]:
MODEL = "claude-opus-5-5"
EFFORT = "low"        # classification needs little reasoning; low keeps cost and time down
MAX_TOKENS = 2048     # the model's internal reasoning counts toward this limit, the answer itself is one word
WORKERS = 8           # requests sent in parallel (lower to 4 if the run is much slower than expected)
QUICK_TEST = True
PRICE_IN, PRICE_OUT = 4.00, 20.00   # US dollars per million tokens for Claude Opus 5.5
CMAP = "Blues"
SEED = 42

RUN_TAG = "_quick" if QUICK_TEST else ""
RESULTS_DIR = BASE_DIR + "/results_llm" + RUN_TAG
os.makedirs(RESULTS_DIR + "/images", exist_ok=True)
os.makedirs(RESULTS_DIR + "/predictions", exist_ok=True)
print("QUICK_TEST =", QUICK_TEST, "| model:", MODEL, "| results folder:", RESULTS_DIR)

## 5. Load the test set (label 0 = bearish, 1 = bullish)

In [ ]:
test = pd.read_csv(DATA_DIR + "/stock_emotions_test.csv")[["text", "label"]]
if QUICK_TEST:
    test = test.sample(n=20, random_state=SEED).reset_index(drop=True)
print(len(test), "test posts | label counts:", test["label"].value_counts().to_dict())
print(test.head(3).to_string())

## 6. The three prompts

Same text as the files in src/llm_prompts (zsp.md, fsp.md, sp.md). The four few-shot examples come from the StockEmotions training set, not the test set.
The prompt is sent as the system prompt; each post is sent as the user message in the form Text: "...".
The structured prompt also asks the API for a fixed JSON answer (label plus a one-sentence reason), so the response is guaranteed to be valid JSON.

In [ ]:
ZERO_SHOT = """# Identity
You are an assistant that labels short financial commentary as either Positive or Negative. You will classify the sentiment of a stock market tweet.

# Instructions
- Only output a single word in your response with no additional formatting or commentary.
- Your response should only be one of the words "Positive" or "Negative" depending on the sentiment of the financial commentary given. No other classification labels are allowed to be used.
- Avoid responding with emojis."""

FEW_SHOT = ZERO_SHOT + """

# Examples
Text: "$AMZN Dow futures up by 100 points already." // Positive
Text: "$AAPL where are all the peeps posting bearish since $200" // Negative
Text: "$TSLA making money 💰 love to short this pig 🐷" // Negative
Text: "$TSLA 400s day is today🤩." // Positive"""

STRUCTURED = """# Context
I want to evaluate the sentiment of a piece of social media financial commentary and label it as either "Positive" or "Negative".

# Objective
Classify the financial commentary as "Positive" (bullish: the writer expects the stock or market to go up, or is pleased about it) or "Negative" (bearish: the writer expects it to go down, or is unhappy about it).

# Examples
Text: "$AMZN Dow futures up by 100 points already." // Positive
Text: "$AAPL where are all the peeps posting bearish since $200" // Negative
Text: "$TSLA making money 💰 love to short this pig 🐷" // Negative
Text: "$TSLA 400s day is today🤩." // Positive

# Output format
Respond with a JSON object with two fields: "label", either "Positive" or "Negative", and "reason", one short sentence of at most 15 words saying what in the text decided the label. No other text."""

STRUCTURED_SCHEMA = {
    "type": "object",
    "properties": {
        "label": {"type": "string", "enum": ["Positive", "Negative"]},
        "reason": {"type": "string"},
    },
    "required": ["label", "reason"],
    "additionalProperties": False,
}

PROMPTS = {"zero_shot": ZERO_SHOT, "few_shot": FEW_SHOT, "structured": STRUCTURED}
for name, p in PROMPTS.items():
    print(f"{name:10} {len(p.split())} words")

## 7. One request to Claude

parse_label turns the answer into 1 (Positive) or 0 (Negative). Anything else (a refusal, a cut-off answer, extra words) becomes None and is later counted as a wrong prediction.

In [ ]:
def parse_label(answer):
    a = answer.strip().strip(".!\"' ").lower()
    if a == "positive":
        return 1
    if a == "negative":
        return 0
    return None


def ask_claude(prompt_name, text):
    kwargs = dict(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=PROMPTS[prompt_name],
        messages=[{"role": "user", "content": f'Text: "{text}"'}],
        output_config={"effort": EFFORT},
    )
    if prompt_name == "structured":
        kwargs["output_config"]["format"] = {"type": "json_schema", "schema": STRUCTURED_SCHEMA}
    response = client.messages.create(**kwargs)

    raw = "".join(b.text for b in response.content if b.type == "text").strip()
    answer, reason = raw, ""
    if response.stop_reason != "end_turn":            # refusal or max_tokens: keep the reason in the raw column
        raw = f"[{response.stop_reason}] " + raw
        answer = ""
    elif prompt_name == "structured":
        try:
            data = json.loads(raw)
            answer, reason = data["label"], data["reason"]
        except (ValueError, KeyError, TypeError):   # not the JSON we asked for: counts as invalid
            answer = ""
    return {"raw": raw, "pred": parse_label(answer), "reason": reason,
            "input_tokens": response.usage.input_tokens, "output_tokens": response.usage.output_tokens}

## 8. Smoke test: one post, three prompts

In [ ]:
post, true_label = test["text"][0], test["label"][0]
print("post:", post, "| true label:", "bullish" if true_label == 1 else "bearish")
for name in PROMPTS:
    r = ask_claude(name, post)
    print(f"{name:10} -> {r['raw']!r:70} pred={r['pred']} tokens in/out={r['input_tokens']}/{r['output_tokens']}")

## 9. Run one prompt over the whole test set

Writes predictions/NAME.csv (one row per post with the raw answer), a Blues confusion matrix, and returns one row of metrics: the same accuracy, precision, recall, F1 as Section 2, plus macro-F1 and bearish recall as Section 3, plus token counts and cost.

In [ ]:
def run_prompt(name):
    start = time.time()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        rows = list(pool.map(lambda t: ask_claude(name, t), test["text"]))
    minutes = (time.time() - start) / 60

    pred_df = test.copy()
    for col in ["raw", "pred", "reason", "input_tokens", "output_tokens"]:
        pred_df[col] = [r[col] for r in rows]
    invalid = int(pred_df["pred"].isna().sum())
    pred_df["pred_scored"] = pred_df["pred"].fillna(1 - pred_df["label"]).astype(int)   # invalid answers count as wrong
    pred_df["correct"] = (pred_df["pred_scored"] == pred_df["label"]).astype(int)
    pred_df.to_csv(f"{RESULTS_DIR}/predictions/{name}.csv", index=False)

    y, p = pred_df["label"], pred_df["pred_scored"]
    cost = (pred_df["input_tokens"].sum() * PRICE_IN + pred_df["output_tokens"].sum() * PRICE_OUT) / 1e6
    row = {"prompt": name, "model": MODEL, "n_posts": len(pred_df), "invalid": invalid, "minutes": round(minutes, 1),
           "accuracy": accuracy_score(y, p),
           "precision": precision_score(y, p, zero_division=0),
           "recall": recall_score(y, p, zero_division=0),
           "f1": f1_score(y, p, zero_division=0),
           "macro_f1": f1_score(y, p, average="macro", zero_division=0),
           "bearish_recall": recall_score(y, p, pos_label=0, zero_division=0),
           "input_tokens": int(pred_df["input_tokens"].sum()),
           "output_tokens": int(pred_df["output_tokens"].sum()),
           "cost_usd": round(cost, 2)}

    cm = confusion_matrix(y, p, labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["bearish", "bullish"]).plot(cmap=CMAP, colorbar=False)
    plt.title(f"Claude {name.replace('_', '-')} prompt")
    plt.savefig(f"{RESULTS_DIR}/images/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"{name}: macro_f1={row['macro_f1']:.3f} | accuracy={row['accuracy']:.3f} | invalid={invalid} | "
          f"${cost:.2f} | {minutes:.1f} min")
    return row

## 10. Run all three prompts

In [ ]:
results = pd.DataFrame([run_prompt(name) for name in PROMPTS])
results.to_csv(RESULTS_DIR + "/llm_results.csv", index=False)
print(results.round(3).to_string(index=False))

## 11. LLM vs the trained models

Macro-F1 on the same test set. TF-IDF number is the best Section 2 baseline (Logistic Regression, Model B, finance only). BERT number is the Section 3 Strategy 1 mean over three seeds, read from Drive.

In [ ]:
bert = pd.read_csv(BASE_DIR + "/results/bert_results.csv").groupby("strategy")["macro_f1"].mean()
llm = results.set_index("prompt")["macro_f1"]
compare = pd.DataFrame({
    "model": ["Always bullish", "TF-IDF + LR\nfinance only", "BERT\nfinance only",
              "Claude\nzero-shot", "Claude\nfew-shot", "Claude\nstructured"],
    "trained_on": ["nothing", "8,000 finance posts", "8,000 finance posts", "nothing", "4 examples in prompt", "4 examples in prompt"],
    "macro_f1": [0.357, 0.711, bert["S1_domain_only"], llm["zero_shot"], llm["few_shot"], llm["structured"]],
})
print(compare.assign(model=compare["model"].str.replace("\n", " ")).round(3).to_string(index=False))
compare.to_csv(RESULTS_DIR + "/llm_vs_trained.csv", index=False)

plt.figure(figsize=(8, 4))
bars = plt.bar(compare["model"], compare["macro_f1"],
               color=["#c6dbef", "#9ecae1", "#6baed6", "#4292c6", "#2171b5", "#08519c"])
for b in bars:
    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f"{b.get_height():.3f}", ha="center", fontsize=9)
plt.ylim(0, 1)
plt.ylabel("Macro-F1 on finance test set")
plt.title("LLM prompting (no training) vs trained models")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR + "/images/llm_vs_trained.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Error analysis

The structured prompt returns a reason with every label, so its mistakes can be read directly: the post, the true label, and what the model said.

In [ ]:
pred = pd.read_csv(f"{RESULTS_DIR}/predictions/structured.csv")
wrong = pred[pred["correct"] == 0]
print(f"structured prompt: {len(wrong)} wrong out of {len(pred)}")
print("wrong by true label:", wrong["label"].map({0: "bearish", 1: "bullish"}).value_counts().to_dict())
for _, r in wrong.head(15).iterrows():
    print(f"\ntrue = {'bullish' if r['label'] == 1 else 'bearish'} | model said: {r['raw']}")
    print(f"   {r['text']}")

## 13. Files written to Drive

In [ ]:
for root, dirs, files in os.walk(RESULTS_DIR):
    for f in sorted(files):
        print(os.path.join(root, f).replace(BASE_DIR + "/", ""))